# StyleMatch · publication-grade method table

Produces a directly comparable Part 4 table and figure from one frozen source-heldout split. The evaluation unit is an independent source; results are macro-averaged by author-language profile. Reported measures are MRR and Recall@1/3/5 with profile-bootstrap 95% confidence intervals. F1 is intentionally omitted because this is ranked retrieval, not binary classification.

In [ ]:
from google.colab import drive
from pathlib import Path
import json, os, shutil, subprocess, sys

drive.mount('/content/drive')
REPO = Path('/content/drive/MyDrive/style_matching')
assert (REPO / 'scripts/export_method_performance.py').exists(), 'Pull the commit containing this notebook and export_method_performance.py'
os.chdir(REPO)

def run(cmd):
    print('>>>', ' '.join(map(str, cmd)), flush=True)
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    if code:
        raise RuntimeError(f'command failed with exit {code}: {" ".join(map(str, cmd))}')

run([sys.executable, '-m', 'pip', 'install', '-q', 'pandas', 'pyarrow', 'sentence-transformers==5.1.1', 'transformers>=4.41,<5', 'scikit-learn', 'matplotlib'])

## 1 · Freeze and verify the evaluation population

Do not rebuild splits in this notebook. Every method must score the same chunk IDs, profile order, dev sources, and locked test sources.

In [ ]:
import pandas as pd

HELDOUT = REPO / 'data/all/meta/all_source_heldout_splits.parquet'
assert HELDOUT.exists(), 'Run the multilingual source/split notebook first'
heldout = pd.read_parquet(HELDOUT)
assert {'train', 'dev', 'test'} <= set(heldout['split'].astype(str))
assert 'independent_source_id' in heldout.columns
snapshot = {
    'rows': len(heldout),
    'profiles': int(heldout.groupby(['language', 'author_or_speaker']).ngroups),
    'independent_sources': int(heldout.groupby(['corpus', 'language', 'author_or_speaker', 'independent_source_id']).ngroups),
    'split_counts': heldout['split'].value_counts().sort_index().to_dict(),
}
print(json.dumps(snapshot, indent=2))

## 2 · Reuse or create aligned score matrices

This cell never fine-tunes. It reuses aligned score matrices first; only a missing matrix requires its finished local model. Model files split by an earlier pull backup are recovered automatically.

In [ ]:
EXP = REPO / 'artifacts/method_table_v1'
EXP.mkdir(parents=True, exist_ok=True)
models = {
    'mstyle_pretrained': ('StyleDistance/mstyledistance', 'd66ed25e48225a503b21a65bc804caf06c886f96'),
    'challenger_pretrained': ('Blablablab/multilingual-style-representation', 'b0147bbf450424fe72c8525fcc02e2e39e3a4024'),
    'mstyle_finetuned': ('mstyledistance_stylematch_v1', None),
    'challenger_finetuned': ('multilingual_author_style_v1', None),
}

def local_model_ready(path):
    path = Path(path)
    weights = list(path.glob('*.safetensors')) + list(path.glob('*.bin'))
    return (path / 'modules.json').exists() and (path / 'config.json').exists() and (path / 'tokenizer_config.json').exists() and bool(weights)

def recovery_roots():
    drive_root = REPO.parent
    return [
        *sorted(drive_root.glob('style_matching_pull_backup*')),
        drive_root / 'style matching pull back',
        drive_root / 'style_matching_pull_back',
        drive_root / 'runtime',
        drive_root / 'stylematch_v1',
    ]

def recover_local_model(dirname):
    target = REPO / 'artifacts' / dirname
    candidates = []
    for root in recovery_roots():
        candidates.extend([
            root / 'artifacts' / dirname,
            root / dirname,
            root / 'style_matching' / 'artifacts' / dirname,
            root / 'style_matching' / dirname,
        ])
    for source in candidates:
        if not source.is_dir():
            continue
        target.mkdir(parents=True, exist_ok=True)
        for src in source.rglob('*'):
            if src.is_file():
                dst = target / src.relative_to(source)
                if not dst.exists():
                    dst.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(src, dst)
    if not local_model_ready(target):
        searched = '\n  '.join(str(p) for p in [target, *candidates])
        raise FileNotFoundError(
            f'Missing complete finished model {dirname}. Expected config, tokenizer, modules, '
            f'and model weights. Searched:\n  {searched}\n'
            'Restore this model from the previous multilingual-training runtime; this notebook does not retrain it.'
        )
    return str(target)

def find_prior_score(label, filename, preferred):
    artifact_roots = [REPO / 'artifacts']
    for root in recovery_roots():
        artifact_roots.extend([root / 'artifacts', root / 'style_matching' / 'artifacts'])
    candidates = [preferred]
    for artifacts_root in artifact_roots:
        candidates.extend([
            artifacts_root / f'eval_{label}' / filename,
            artifacts_root / 'method_exploration_v1' / f'eval_{label}' / filename,
            artifacts_root / 'method_table_v1' / f'eval_{label}' / filename,
        ])
    return next((path for path in candidates if path.exists()), preferred)

# Resolve every missing local dependency before starting any new GPU encoding.
score_files = {}
resolved_models = {}
for label, (model_name, _) in models.items():
    preferred = EXP / f'eval_{label}' / 'style_embedding_scores.npz'
    score_files[label] = find_prior_score(label, 'style_embedding_scores.npz', preferred)
    if not score_files[label].exists() and label.endswith('_finetuned'):
        resolved_models[label] = recover_local_model(model_name)
    else:
        resolved_models[label] = model_name
    if score_files[label].exists():
        print(f'reusing {label}: {score_files[label]}')

score_specs = []
for label, (model_name, revision) in models.items():
    out_dir = EXP / f'eval_{label}'
    score_file = score_files[label]
    if not score_file.exists():
        model_name = resolved_models[label]
        cmd = [sys.executable, 'scripts/style_embedding_recall.py', '--input', str(HELDOUT), '--out-dir', str(out_dir), '--model-name', model_name, '--batch-size', '128', '--train-cap', '300', '--eval-splits', 'dev,test', '--device', 'cuda']
        if revision:
            cmd.extend(['--model-revision', revision])
        run(cmd)
        score_file = out_dir / 'style_embedding_scores.npz'
    score_specs.append(f'{label}_centroid={score_file}:single_centroid_scores')
    if label == 'mstyle_finetuned':
        score_specs.append(f'{label}_prototype={score_file}:source_prototype_scores')

classical_dir = EXP / 'eval_classical'
classical_scores = find_prior_score('classical', 'style_robust_scores.npz', classical_dir / 'style_robust_scores.npz')
if not classical_scores.exists():
    run([sys.executable, 'scripts/style_robust_baseline.py', '--input', str(HELDOUT), '--out-dir', str(classical_dir)])
    classical_scores = classical_dir / 'style_robust_scores.npz'
score_specs.append(f'classical_style={classical_scores}:style_only_fusion')
print('aligned views:', *score_specs, sep='\n  ' )

## 3 · Locked source-level comparison and 95% intervals

In [ ]:
comparison_dir = EXP / 'comparison'
cmd = [sys.executable, 'scripts/evaluate_multiview_fusion.py', '--input', str(HELDOUT), '--output-dir', str(comparison_dir), '--bootstrap-runs', '5000', '--seed', '20260722']
for spec in score_specs:
    cmd.extend(['--scores', spec])
run(cmd)
report_path = comparison_dir / 'multiview_fusion_metrics.json'
run([sys.executable, 'scripts/export_method_performance.py', '--input', str(report_path), '--out-dir', str(EXP)])

In [ ]:
from IPython.display import Image, display

table = pd.read_csv(EXP / 'method_performance.csv')
display(table[['test_mrr_rank', 'method', 'n_test_sources', 'n_test_profiles', 'mrr', 'mrr_ci_low', 'mrr_ci_high', 'recall_at_1', 'recall_at_3', 'recall_at_3_ci_low', 'recall_at_3_ci_high', 'recall_at_5', 'selected']].style.format(precision=3))
display(Image(filename=str(EXP / 'method_performance.png')))
return_files = [EXP / 'method_performance.csv', EXP / 'method_performance.json', EXP / 'method_performance.png', EXP / 'method_performance.pdf', report_path]
share_dir = EXP / 'share'
share_dir.mkdir(exist_ok=True)
for path in return_files:
    shutil.copy2(path, share_dir / path.name)
share = shutil.make_archive(str(REPO / 'artifacts/stylematch_method_table_v1'), 'zip', share_dir, '.')
print('RETURN THESE FIVE FILES:')
for path in return_files:
    print(path)
print('optional bundle:', share)